In [13]:
##### Converts raw raster on field size into final varaiable needed 
# pixel and subnational (vector) scale

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [9]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# raw raster
raw = f"{cd}/Data/Raw/Predictors/Field_size_Lesiv/dominant_field_size_categories.tif"

# Save paths
pixels_vlarge = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_vlarge.tif"
pixels_large = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_large.tif"
pixels_medium = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_medium.tif"
pixels_small = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_small.tif"
pixels_vsmall = f"{cd}/Data/Clean/Predictors/Rasters/field_size_share_vsmall.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/field_size_shares.csv"

In [11]:
#### Run resampling for pixel scale 

### PREP 

# set list of field size categories 
categories = {
    "vlarge": (3502, pixels_vlarge),
    "large":  (3503, pixels_large),
    "medium": (3504, pixels_medium),
    "small":  (3505, pixels_small),
    "vsmall": (3506, pixels_vsmall),
}

# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### RESAMPLE 
with rasterio.open(raw) as src:
    data          = src.read(1)
    src_nodata    = src.nodata
    src_transform = src.transform
    src_crs       = src.crs

    # mask out both nodata AND no-field (0) so the shares are just of areas with fields 
    exclude_mask = (data == 0)
    if src_nodata is not None:
        exclude_mask |= (data == src_nodata)

    for name, (code, out_path) in categories.items():
        # 1.0 where this category, 0.0 for other field categories, nan for no-field/nodata
        # average of this binary over field-only pixels = share of this category among fields
        binary = np.where(exclude_mask, np.nan, (data == code).astype(np.float32))

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=binary,
            destination=dst_array,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.average,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )

        out_meta = dst_meta.copy()
        out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

        out_arr = dst_array.copy()
        out_arr[np.isnan(out_arr)] = -9999

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_arr, 1)

        print(f"Saved {name} → {out_path}")

Saved vlarge → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_vlarge.tif
Saved large → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_large.tif
Saved medium → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_medium.tif
Saved small → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_small.tif
Saved vsmall → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/field_size_share_vsmall.tif


In [14]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
with rasterio.open(raw) as src:
    raster_crs = src.crs
    data = src.read(1)
    src_transform = src.transform
    src_nodata = src.nodata

gdf_proj = total_geo.to_crs(raster_crs)

# mask out both nodata and no-field (0) pixels by setting them to a single nodata value
clean = data.copy().astype(np.float32)
clean[(data == 0)] = np.nan
if src_nodata is not None:
    clean[(data == src_nodata)] = np.nan
clean[(data == -9999)] = np.nan


### Run zonal stats
# get pixel counts per category per polygon
zonal = rasterstats.zonal_stats(
    gdf_proj,
    clean,
    affine=src_transform,
    categorical=True,
    nodata=np.nan
)

### Convert to df

# map raster codes to column names
code_map = {
    3502: "share_vlarge_field",
    3503: "share_large_field",
    3504: "share_medium_field",
    3505: "share_small_field",
    3506: "share_vsmall_field",
}

# get project ID's
result = total_geo[["PROJ_ID"]].copy()

# add in zonal stats
for code, col_name in code_map.items():
    counts = [z.get(code, 0) for z in zonal]
    totals = [sum(z.get(c, 0) for c in code_map.keys()) for z in zonal]
    result[col_name] = [
        c / t if t > 0 else None
        for c, t in zip(counts, totals)
    ]

### Save
result.to_csv(vectors, index=False)